# Demo - Anthropic SDK: `tool_choice`

| setting | meaning |
|---|---|
| `auto` | the model decides — *default when tools are provided* |
| `any` | must call **some** tool (it still picks which) |
| `tool` | must call **this** tool (you pick, it obeys) |
| `none` | must not call any tool — *default when no tools are provided* |

## Setup

Install the Anthropic SDK package and dotenv so we can import an environment variable with our Anthropic API key

In [ ]:
!uv pip install anthropic dotenv

Import the `anthropic` package, create an API client, and define the model

In [ ]:
import json
import anthropic

MODEL = "claude-sonnet-4-6"

from dotenv import load_dotenv, find_dotenv

client = anthropic.Anthropic()

Define the query (prompt) for the agent, and a system prompt that explicitly asks for a preamble before any tool call.

The system prompt is the control for the whole demo: it never changes, so any difference in whether a preamble appears is caused by `tool_choice` alone.

In [ ]:
QUERY = "What's the weather in Canberra right now?"

SYSTEM = (
    "Before you call a tool, tell the user in one short sentence what you are about "
    "to do and why. Always do this, even when a tool call is required of you."
)

Define the tools available to the agent.

In [ ]:
# Two tools, deliberately unrelated with rich descriptions defining their purpose
TOOLS = [
    {
        "name": "get_weather",
        "description": (
            "Retrieves the current weather observation for a named city: "
            "temperature in degrees Celsius, conditions, and relative humidity. "
            "Use this whenever the user asks what the weather is doing right now "
            "in a specific place. Does not return forecasts — current conditions only."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. 'Canberra'."}},
            "required": ["city"],
        },
    },
    {
        "name": "create_ticket",
        "description": (
            "Opens a new ticket in the internal support queue and returns its "
            "reference number. Use this only when the user is reporting a problem "
            "that needs to be tracked and actioned by a human. Never use it to "
            "answer a question — it creates work, it does not retrieve information."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"summary": {"type": "string", "description": "One-line summary of the issue."}},
            "required": ["summary"],
        },
    },
]

## Helper function

This helper function is just a harness for the purpose of this demo to show what happens when the agent is repeatedly invoked with the only difference being the value of `tool_choice`. 

In [ ]:
def run(choice, query=QUERY, quiet=False):
    """`choice` is the only thing that ever changes."""
    resp = client.messages.create(
        model=MODEL,
        max_tokens=300,
        system=SYSTEM,
        tools=TOOLS,
        tool_choice=choice,
        messages=[{"role": "user", "content": query}],
    )

    text = [b.text.strip() for b in resp.content if b.type == "text" and b.text.strip()]
    calls = [b for b in resp.content if b.type == "tool_use"]
    forced = choice["type"] in ("any", "tool")

    if not quiet:
        print(f"tool_choice : {json.dumps(choice)}")
        print(f"stop_reason : {resp.stop_reason}")
        if calls:
            if text:
                preamble = f"“{_short(text[0])}”"
            elif forced:
                preamble = "(none — the API prefilled the assistant turn)"
            else:
                preamble = "(none — nothing forced it; the model just didn't write one)"
            print(f"preamble    : {preamble}")
            for b in calls:
                print(f"tool call   : {b.name}({json.dumps(b.input)})")
        else:
            print("tool call   : (none — the tools were defined but unusable)")
            if text:
                print(f"text        : “{_short(text[0])}”")

    name = calls[0].name if calls else None
    return name if quiet else None   # loud mode returns nothing — no stray Out[] under the beat


def _short(s, width=72):
    s = " ".join(s.split())
    return s if len(s) <= width else s[: width - 1] + "…"

---
## 1 · `auto` — the model decides

The default when tools are provided. Watch two things: it chose to call a tool,
**and it said so first** — the system prompt asked for a preamble and nothing
stopped it from writing one.

Under `auto` the preamble is a prompted behaviour, not a guarantee. Without that
instruction the model often skips straight to the `tool_use` block.

In [ ]:
run({"type": "auto"})

---
## 2 · `any` — must call some tool

Same tool as last time. Same system prompt, still asking for a preamble. Now look
at the preamble line.

It's gone — and that is not the model being terse. With `any` and `tool` the API
**prefills the assistant turn** to force the call, so no natural-language
explanation can precede the `tool_use` block. From the docs: the model "will not
emit a natural language response or explanation before `tool_use` content
blocks, *even if explicitly asked to do so*." We asked. It couldn't.

In [ ]:
run({"type": "any"})

---
## 3 · `tool` — must call this tool

Pinned to `create_ticket`. The user asked about the weather.

In [ ]:
run({"type": "tool", "name": "create_ticket"})

It opened a support ticket about a weather question.

**Forcing a tool call is not the same as making one appropriate.** `tool` is how
you enforce order — extract before enrich — not how you improve judgement.

---
## 4 · `none` — must not call any tool

The tools are still in the request. Still parsed, still billed. Just unusable.

In [ ]:
run({"type": "none"})